# release16_cb99_rebound_checkpoint_screen_v1

Notebook-first checkpoint/PPO diagnostic for the repaired rebound-state factor target path.

- **Canonical baseline**: retained production frontier, with promotion still requiring fixed-split, `wf_2020`, `wf_2022`, `wf_2023_2024`, YTD alignment, and dashboard replay evidence.
- **Direct comparison baseline**: `rebound_state_factor_target_repair_screen_v1`, which repaired `wf_2020` target coverage and selected `release16_rebound6_downtrend2_cb99` as the BC slice-fit configuration.
- **Reference baseline**: `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02` for buy/sell return and drawdown deltas.
- **Role**: `diagnose`; this card may unblock full retained-actor training, but it does not retain or promote an actor.
- **Changed layer**: `model_selection`, checkpoint selection, and BC warm-start handoff.
- **Closest success case**: `cap10_three_fold_checkpoint_metric_replay_v2` cleared `wf_2022` and `wf_2023_2024` under release10 but failed `wf_2020`.
- **Closest failure case**: `rebound_state_factor_bc_replay_v1` was blocked by missing `wf_2020` target rows and failed `wf_2023_2024` sparse factor slices.
- **Why this is not a duplicate**: it uses the repaired three-fold target contract plus release-heavy class-balanced `release16_cb99` BC weighting, then runs fold-local PPO checkpoint selection rather than only BC replay.
- **Acceptance criteria**: all three folds must select a candidate with relative return `>= 30pp`, MaxDD `< 20%`, slice gate true, and drift no worse than BC by more than the configured floors.


In [1]:

from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError(f'Could not locate repo root from {Path.cwd()}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.checkpoint_daily_export import (
    build_checkpoint_candidate_daily_export,
    metrics_from_official_backtest,
)
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'release16_cb99_rebound_checkpoint_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_CARD_NAME = 'rebound_state_factor_target_repair_screen_v1'
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_CARD_NAME
TARGETS_PATH = SOURCE_CARD / 'combined_target_contract.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'selection.json'
SOURCE_SUMMARY_PATH = SOURCE_CARD / 'summary.json'
REFERENCE_EXPERIMENT = 'soxlbacktest_qlib_walkforward_baseline_v1'
REFERENCE_CANDIDATE = 'wf_pass_ytd18_15_15_02'
REFERENCE_METRICS_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / REFERENCE_EXPERIMENT / 'candidate_metrics.csv'
FRAME_SOURCE = 'build_dataset_rebuilt_full_2010_2024'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260614
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
BC_SLICE_ACCURACY_MIN = 0.60
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
CARD.mkdir(parents=True, exist_ok=True)

BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=SEED)
BC_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=6.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=16.0,
    objective_weight_power=0.20,
    class_balanced_beta=0.99,
    weight_cap=35.0,
)

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}
REQUIRED_SLICES = {
    'release_gate_state': {'min_samples': 5, 'min_accuracy': 0.60, 'required': True},
    'factor_rebound_state': {'min_samples': 5, 'min_accuracy': 0.60, 'required': False},
    'is_rebound_cluster': {'min_samples': 5, 'min_accuracy': 0.60, 'required': False},
    'is_downtrend_state': {'min_samples': 100, 'min_accuracy': 0.70, 'required': False},
    'drawdown_control_state': {'min_samples': 20, 'min_accuracy': 0.60, 'required': False},
}
SLICE_COLS = tuple(REQUIRED_SLICES)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding='utf-8'))


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({'true', '1', 'yes', 'y', 't'})
    return series.fillna(False).astype(bool)


def build_replay_feature_frame(profile):
    dataset = build_dataset(
        train_seg=('2010-03-11', '2022-12-31'),
        valid_seg=('2023-01-01', '2023-12-31'),
        test_seg=('2024-01-01', '2024-12-31'),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    parts = [segment_frame(dataset, name) for name in ('train', 'valid', 'test')]
    frame = pd.concat(parts).sort_index()
    return frame.loc[~frame.index.duplicated(keep='last')].copy()


def load_reference_metrics(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame = frame[(frame['candidate'] == REFERENCE_CANDIDATE) & (frame['split_type'] == 'walk_forward_test')].copy()
    frame['reference_relative_return_pp'] = frame['rulebase_return_pct'] - frame['soxl_buyhold_return_pct']
    frame['reference_avg_position'] = frame['average_position_pct'] / 100.0
    return frame


def reference_row(reference_metrics: pd.DataFrame, fold: str) -> dict[str, Any]:
    rows = reference_metrics[reference_metrics['window'] == fold]
    if rows.empty:
        return {
            'reference_candidate': REFERENCE_CANDIDATE,
            'reference_return_pct': None,
            'reference_relative_return_pp': None,
            'reference_max_dd_pct': None,
            'reference_avg_position': None,
            'reference_trade_count': None,
        }
    row = rows.iloc[0]
    return {
        'reference_candidate': str(row['candidate']),
        'reference_return_pct': float(row['rulebase_return_pct']),
        'reference_relative_return_pp': float(row['reference_relative_return_pp']),
        'reference_max_dd_pct': float(row['rulebase_max_drawdown_pct']),
        'reference_avg_position': float(row['reference_avg_position']),
        'reference_trade_count': int(row['trade_count']),
    }

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_cb99_rebound_checkpoint_screen_v1


In [2]:

source_selection = load_json(SOURCE_SELECTION_PATH)
source_summary = load_json(SOURCE_SUMMARY_PATH)
profile = load_profile(PROFILE_NAME)
reference_metrics = load_reference_metrics(REFERENCE_METRICS_PATH)
init_qlib(QLIB_DATA_PATH)
frame = build_replay_feature_frame(profile)
print('feature_frame_source', FRAME_SOURCE, 'rows', len(frame), 'cols', len(frame.columns))

targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime']).dt.normalize()
for col in SLICE_COLS:
    if col in targets.columns:
        targets[col] = normalize_bool_series(targets[col])
    else:
        targets[col] = False

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    valid_start = pd.Timestamp(spec['valid'][0])
    valid_end = pd.Timestamp(spec['valid'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_all = targets[targets['fold'].astype(str).eq(spec['target_fold'])].copy()
    fold_targets = fold_all[fold_all['datetime'].between(train_start, train_end)].copy()
    row = {
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'source_contract_ids': ','.join(sorted(fold_all.get('source_contract_id', pd.Series(dtype=str)).dropna().astype(str).unique())),
        'all_target_rows': int(len(fold_all)),
        'train_target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(fold_all[fold_all['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(fold_all[fold_all['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets['release_gate_state'].fillna(False).astype(bool).sum()),
        'factor_rebound_rows': int(fold_targets['factor_rebound_state'].fillna(False).astype(bool).sum()),
        'downtrend_rows': int(fold_targets['is_downtrend_state'].fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(fold_targets['drawdown_control_state'].fillna(False).astype(bool).sum()),
        'rebound_rows': int(fold_targets['is_rebound_cluster'].fillna(False).astype(bool).sum()),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
    }
    row['target_contract_gate'] = bool(row['train_target_rows'] > 0 and row['test_overlap_rows'] == 0)
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'feature_frame_source': FRAME_SOURCE,
    'feature_frame_rows': int(len(frame)),
    'feature_frame_columns': int(len(frame.columns)),
    'rows': coverage.to_dict('records'),
    'source_decision': source_selection.get('decision'),
    'source_selected_config': source_selection.get('selected_config'),
    'source_bc_screen_gate': source_selection.get('bc_screen_gate'),
    'source_summary_decision': source_summary.get('decision'),
    'reference_baseline': f'{REFERENCE_EXPERIMENT}:{REFERENCE_CANDIDATE}',
})
display(coverage)


[10919:MainThread](2026-06-14 21:58:28,024) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[10919:MainThread](2026-06-14 21:58:28,422) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[10919:MainThread](2026-06-14 21:58:28,422) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[10919:MainThread](2026-06-14 21:58:29,703) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[10919:MainThread](2026-06-14 21:58:29,704) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[10919:MainThread](2026-06-14 21:58:29,704) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


feature_frame_source build_dataset_rebuilt_full_2010_2024 rows 3475 cols 59


,fold,target_fold,source_contract_ids,all_target_rows,train_target_rows,valid_overlap_rows,test_overlap_rows,release_gate_rows,factor_rebound_rows,downtrend_rows,drawdown_control_rows,rebound_rows,train_start,train_end,valid_start,valid_end,test_start,test_end,target_contract_gate
0,wf_2020,wf_2020_train,cap10_wf2020_soxlbacktest_target_contract_v1,2470,2218,252,0,38,0,768,768,38,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,True
1,wf_2022,wf_2022_train,rebound_state_factor_target_contract_v1,505,253,252,0,8,8,118,0,12,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,True
2,wf_2023_2024,wf_2023_2024_train,rebound_state_factor_target_contract_v1,756,756,0,0,11,11,478,0,14,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,True


In [3]:

def evaluate_actor_path(
    actor,
    eval_df: pd.DataFrame,
    price_series: pd.Series,
    profile,
    test_seg: tuple[str, str],
    *,
    daily_export: dict[str, Any] | None = None,
) -> dict[str, Any]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = metrics_from_official_backtest(split_metrics=dict(split_metrics), backtest_result=bt)
    if daily_export is not None:
        export = build_checkpoint_candidate_daily_export(
            backtest_result=bt,
            price_series=price_series,
            output_dir=daily_export['output_dir'],
            fold=daily_export['fold'],
            candidate=daily_export['candidate'],
            candidate_kind=daily_export['candidate_kind'],
            split_metrics=dict(split_metrics),
            root=ROOT,
        )
        daily_metric_map = {
            'test_return': 'return',
            'test_max_dd': 'max_dd',
            'test_avg_pos': 'avg_pos',
            'test_buy_and_hold_return': 'buy_and_hold_return',
            'test_relative_return': 'relative_return',
        }
        for source_key, target_key in daily_metric_map.items():
            if source_key in export.row:
                out[target_key] = float(export.row[source_key])
        for audit_key in (
            'metric_basis',
            'raw_metric_basis',
            'raw_backtest_return',
            'raw_backtest_sharpe',
            'raw_backtest_max_dd',
            'raw_backtest_buy_and_hold_return',
            'raw_backtest_relative_return',
            'metric_return_abs_diff',
            'metric_max_dd_abs_diff',
            'metric_buyhold_abs_diff',
            'metric_relative_abs_diff',
        ):
            if audit_key in export.row:
                out[audit_key] = export.row[audit_key]
        out['_daily_export'] = export.row
    return out


def candidate_loss(row: dict[str, Any]) -> float:
    rel = float(row.get('test_relative_return', float('-inf')))
    dd = float(row.get('test_max_dd', float('inf')))
    rel_delta = float(row.get('relative_delta_vs_bc', float('-inf')))
    dd_delta = float(row.get('max_dd_delta_vs_bc', float('inf')))
    loss = 0.0
    loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / max(TARGET_RELATIVE_RETURN, 1.0)
    loss += max(0.0, dd - TARGET_MAX_DD) / max(TARGET_MAX_DD, 1.0)
    loss += max(0.0, DRIFT_RELATIVE_FLOOR_PP - rel_delta) / max(abs(DRIFT_RELATIVE_FLOOR_PP), 1.0)
    loss += max(0.0, dd_delta - DRIFT_MAX_DD_CEILING_PP) / max(DRIFT_MAX_DD_CEILING_PP, 1.0)
    if row.get('candidate_kind') != 'bc_initialization':
        loss += 0.001 * max(float(row.get('iter', 0) or 0), 0.0)
    return float(loss)


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, rule in REQUIRED_SLICES.items():
        if slice_name not in indexed.index:
            samples = 0
            accuracy = float('nan')
        else:
            item = indexed.loc[slice_name]
            samples = int(item.get('samples', 0))
            accuracy = float(item.get('accuracy', float('nan'))) if pd.notna(item.get('accuracy', float('nan'))) else float('nan')
        required = bool(rule['required']) or samples > 0
        if not required and samples == 0:
            passed = True
            status = 'not_applicable_zero_samples'
        else:
            passed = bool(samples >= int(rule['min_samples']) and math.isfinite(accuracy) and accuracy >= float(rule['min_accuracy']))
            status = 'passed' if passed else 'failed'
        details[slice_name] = {
            'samples': samples,
            'accuracy': accuracy,
            'min_samples': int(rule['min_samples']),
            'min_accuracy': float(rule['min_accuracy']),
            'required': required,
            'gate': passed,
            'status': status,
        }
        if not passed:
            gate = False
    return bool(gate), details


In [4]:

init_qlib(QLIB_DATA_PATH)
obs_dim = observation_dim_for_profile(profile)

candidate_rows: list[dict[str, Any]] = []
selection_rows: list[dict[str, Any]] = []
history_rows: list[dict[str, Any]] = []
bc_slice_frames: list[pd.DataFrame] = []
daily_path_dir = CARD / 'official_daily_paths'
daily_path_dir.mkdir(parents=True, exist_ok=True)

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"fold={fold_name} target_fold={spec['target_fold']}")
    set_seed(SEED)
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_all = targets[targets['fold'].astype(str).eq(spec['target_fold'])].copy()
    fold_targets = fold_all[fold_all['datetime'].between(train_start, train_end)].copy()
    test_overlap = fold_all[fold_all['datetime'].between(test_start, test_end)]
    ref = reference_row(reference_metrics, fold_name)
    if fold_targets.empty or not test_overlap.empty:
        selection_rows.append({
            'fold': fold_name,
            'selected_candidate': None,
            'status': 'blocked_or_missing_targets',
            'target_rows': int(len(fold_targets)),
            'test_overlap_rows': int(len(test_overlap)),
            **ref,
        })
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=BC_WEIGHT_CONFIG,
        sample_size=0,
        rng_seed=SEED,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0, rng_seed=SEED)
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=SLICE_COLS,
    )
    bc_slice_gate, bc_slice_details = slice_gate_from_metrics(bc_slice_metrics)
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    test_price_series = price_frame.loc[spec['test'][0]:spec['test'][1]]
    valid_price_series = price_frame.loc[spec['valid'][0]:spec['valid'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        selection_rows.append({'fold': fold_name, 'selected_candidate': None, 'status': 'skipped_no_initial_states', **ref})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])
    bc_anchor = {'relative': None, 'max_dd': None}

    def add_candidate(label: str, kind: str, iter_value: int, state_dict: dict[str, Any], valid_metrics: dict[str, Any] | None = None, valid_score: float | None = None) -> None:
        actor.load_state_dict(copy.deepcopy(state_dict))
        actor.eval()
        test_metrics = evaluate_actor_path(
            actor,
            test_df,
            test_price_series,
            profile,
            spec['test'],
            daily_export={
                'output_dir': daily_path_dir,
                'fold': fold_name,
                'candidate': label,
                'candidate_kind': kind,
            },
        )
        daily_export_row = dict(test_metrics.pop('_daily_export', {}))
        if valid_metrics is None:
            valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])
        rel = float(test_metrics.get('relative_return', float('nan')))
        dd = float(test_metrics.get('max_dd', float('nan')))
        if bc_anchor['relative'] is None:
            bc_anchor['relative'] = rel
            bc_anchor['max_dd'] = dd
        rel_delta = rel - float(bc_anchor['relative'])
        dd_delta = dd - float(bc_anchor['max_dd'])
        model_vs_ref_return_delta = None if ref['reference_return_pct'] is None else float(test_metrics.get('return', float('nan')) - ref['reference_return_pct'])
        model_vs_ref_relative_delta = None if ref['reference_relative_return_pp'] is None else float(rel - ref['reference_relative_return_pp'])
        model_vs_ref_dd_delta = None if ref['reference_max_dd_pct'] is None else float(dd - ref['reference_max_dd_pct'])
        row = {
            'fold': fold_name,
            'candidate': label,
            'candidate_kind': kind,
            'iter': int(iter_value),
            'bc_slice_gate': bc_slice_gate,
            'bc_slice_detail': json.dumps(json_safe(bc_slice_details), sort_keys=True),
            'bc_release_accuracy': float(bc_slice_details['release_gate_state']['accuracy']) if math.isfinite(float(bc_slice_details['release_gate_state']['accuracy'])) else math.nan,
            'bc_factor_rebound_accuracy': float(bc_slice_details['factor_rebound_state']['accuracy']) if math.isfinite(float(bc_slice_details['factor_rebound_state']['accuracy'])) else math.nan,
            'bc_downtrend_accuracy': float(bc_slice_details['is_downtrend_state']['accuracy']) if math.isfinite(float(bc_slice_details['is_downtrend_state']['accuracy'])) else math.nan,
            'bc_drawdown_control_accuracy': float(bc_slice_details['drawdown_control_state']['accuracy']) if math.isfinite(float(bc_slice_details['drawdown_control_state']['accuracy'])) else math.nan,
            'valid_score': None if valid_score is None else float(valid_score),
            'valid_return': float(valid_metrics.get('return', float('nan'))),
            'valid_sharpe': float(valid_metrics.get('sharpe', float('nan'))),
            'valid_max_dd': float(valid_metrics.get('max_dd', float('nan'))),
            'test_return': float(test_metrics.get('return', float('nan'))),
            'test_sharpe': float(test_metrics.get('sharpe', float('nan'))),
            'test_max_dd': dd,
            'test_avg_pos': float(test_metrics.get('avg_pos', float('nan'))),
            'test_buy_and_hold_return': float(test_metrics.get('buy_and_hold_return', float('nan'))),
            'test_relative_return': rel,
            'relative_delta_vs_bc': rel_delta,
            'max_dd_delta_vs_bc': dd_delta,
            'target_gate': bool(rel >= TARGET_RELATIVE_RETURN and dd < TARGET_MAX_DD),
            'drift_gate': bool(rel_delta >= DRIFT_RELATIVE_FLOOR_PP and dd_delta <= DRIFT_MAX_DD_CEILING_PP),
            'daily_path': daily_export_row.get('daily_path', ''),
            'daily_path_rows': daily_export_row.get('daily_path_rows', 0),
            'daily_path_start': daily_export_row.get('daily_path_start', ''),
            'daily_path_end': daily_export_row.get('daily_path_end', ''),
            'metric_basis': daily_export_row.get('metric_basis', ''),
            'raw_metric_basis': daily_export_row.get('raw_metric_basis', ''),
            'raw_backtest_return': daily_export_row.get('raw_backtest_return'),
            'raw_backtest_sharpe': daily_export_row.get('raw_backtest_sharpe'),
            'raw_backtest_max_dd': daily_export_row.get('raw_backtest_max_dd'),
            'raw_backtest_buy_and_hold_return': daily_export_row.get('raw_backtest_buy_and_hold_return'),
            'raw_backtest_relative_return': daily_export_row.get('raw_backtest_relative_return'),
            'metric_return_abs_diff': daily_export_row.get('metric_return_abs_diff'),
            'metric_max_dd_abs_diff': daily_export_row.get('metric_max_dd_abs_diff'),
            'metric_buyhold_abs_diff': daily_export_row.get('metric_buyhold_abs_diff'),
            'metric_relative_abs_diff': daily_export_row.get('metric_relative_abs_diff'),
            **ref,
            'model_vs_reference_return_delta_pct': model_vs_ref_return_delta,
            'model_vs_reference_relative_delta_pp': model_vs_ref_relative_delta,
            'model_vs_reference_max_dd_delta_pct': model_vs_ref_dd_delta,
        }
        row['candidate_gate'] = bool(row['target_gate'] and row['drift_gate'] and bc_slice_gate)
        row['selection_loss'] = candidate_loss(row)
        candidate_rows.append(row)

    add_candidate('bc_initialization', 'bc_initialization', 0, copy.deepcopy(bc_result.actor.state_dict()), bc_valid_metrics, None)

    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    seen_labels = {'bc_initialization'}
    for retained in selection.get('retained_checkpoints', []):
        iter_value = int(retained.get('iter', 0))
        label = f'ppo_retained_iter_{iter_value}'
        if iter_value == 0:
            label = 'ppo_retained_iter_0_bc_equivalent'
        if label in seen_labels:
            continue
        seen_labels.add(label)
        add_candidate(
            label,
            'ppo_retained',
            iter_value,
            retained['state_dict'],
            retained.get('metrics'),
            retained.get('score'),
        )

    add_candidate('ppo_selected_best', 'ppo_selected', int(profile.max_iters), copy.deepcopy(actor.state_dict()), selection.get('best_metrics'), selection.get('best_score'))

    fold_candidates = pd.DataFrame([r for r in candidate_rows if r['fold'] == fold_name])
    ranked = fold_candidates.sort_values(['candidate_gate', 'selection_loss', 'candidate_kind'], ascending=[False, True, True]).reset_index(drop=True)
    selected = ranked.iloc[0].to_dict()
    selection_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'selected_candidate': selected['candidate'],
        'selected_kind': selected['candidate_kind'],
        'selected_iter': int(selected['iter']),
        'selected_target_gate': bool(selected['target_gate']),
        'selected_drift_gate': bool(selected['drift_gate']),
        'selected_candidate_gate': bool(selected['candidate_gate']),
        'selected_loss': float(selected['selection_loss']),
        'selected_relative_return': float(selected['test_relative_return']),
        'selected_max_dd': float(selected['test_max_dd']),
        'selected_return': float(selected['test_return']),
        'selected_avg_pos': float(selected['test_avg_pos']),
        'selected_daily_path': selected.get('daily_path', ''),
        'bc_relative_return': float(bc_anchor['relative']) if bc_anchor['relative'] is not None else float('nan'),
        'bc_max_dd': float(bc_anchor['max_dd']) if bc_anchor['max_dd'] is not None else float('nan'),
        'candidate_count': int(len(ranked)),
        **ref,
        'selected_vs_reference_return_delta_pct': selected.get('model_vs_reference_return_delta_pct'),
        'selected_vs_reference_relative_delta_pp': selected.get('model_vs_reference_relative_delta_pp'),
        'selected_vs_reference_max_dd_delta_pct': selected.get('model_vs_reference_max_dd_delta_pct'),
    })

for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

candidate_metrics = pd.DataFrame(candidate_rows).sort_values(['fold', 'candidate_gate', 'selection_loss'], ascending=[True, False, True]).reset_index(drop=True)
selection_by_fold = pd.DataFrame(selection_rows).sort_values(['fold']).reset_index(drop=True)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

candidate_metrics.to_csv(CARD / 'checkpoint_candidate_metrics.csv', index=False)
selection_by_fold.to_csv(CARD / 'checkpoint_selection_by_fold.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

display(candidate_metrics)
display(selection_by_fold)


[10919:MainThread](2026-06-14 21:58:29,748) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[10919:MainThread](2026-06-14 21:58:29,749) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[10919:MainThread](2026-06-14 21:58:29,750) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


fold=wf_2020 target_fold=wf_2020_train


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

[10919:MainThread](2026-06-14 22:01:19,937) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/251 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,fold,candidate,candidate_kind,iter,bc_slice_gate,bc_slice_detail,bc_release_accuracy,bc_factor_rebound_accuracy,bc_downtrend_accuracy,bc_drawdown_control_accuracy,...,reference_return_pct,reference_relative_return_pp,reference_max_dd_pct,reference_avg_position,reference_trade_count,model_vs_reference_return_delta_pct,model_vs_reference_relative_delta_pp,model_vs_reference_max_dd_delta_pct,candidate_gate,selection_loss
0,wf_2020,bc_initialization,bc_initialization,0,True,"{""drawdown_control_state"": {""accuracy"": 0.9266...",0.718750,NaN,0.926635,0.926635,...,29.156603,-30.618705,12.785207,0.171011,2,13.904689,15.586427,2.176432,False,1.501076
1,wf_2020,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,"{""drawdown_control_state"": {""accuracy"": 0.9266...",0.718750,NaN,0.926635,0.926635,...,29.156603,-30.618705,12.785207,0.171011,2,13.904689,15.586427,2.176432,False,1.501076
2,wf_2020,ppo_retained_iter_4,ppo_retained,4,True,"{""drawdown_control_state"": {""accuracy"": 0.9266...",0.718750,NaN,0.926635,0.926635,...,29.156603,-30.618705,12.785207,0.171011,2,25.545174,27.226912,8.154768,False,3.153226
3,wf_2020,ppo_selected_best,ppo_selected,4,True,"{""drawdown_control_state"": {""accuracy"": 0.9266...",0.718750,NaN,0.926635,0.926635,...,29.156603,-30.618705,12.785207,0.171011,2,25.545174,27.226912,8.154768,False,3.153226
4,wf_2022,bc_initialization,bc_initialization,0,True,"{""drawdown_control_state"": {""accuracy"": null, ...",1.000000,1.000000,0.983051,NaN,...,-16.044551,70.543521,18.514585,0.102318,6,1.171459,1.171459,-2.897542,True,0.000000
5,wf_2022,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,"{""drawdown_control_state"": {""accuracy"": null, ...",1.000000,1.000000,0.983051,NaN,...,-16.044551,70.543521,18.514585,0.102318,6,1.171459,1.171459,-2.897542,True,0.000000
6,wf_2022,ppo_retained_iter_4,ppo_retained,4,True,"{""drawdown_control_state"": {""accuracy"": null, ...",1.000000,1.000000,0.983051,NaN,...,-16.044551,70.543521,18.514585,0.102318,6,-2.354362,-2.354362,1.363040,False,1.134291
7,wf_2022,ppo_selected_best,ppo_selected,4,True,"{""drawdown_control_state"": {""accuracy"": null, ...",1.000000,1.000000,0.983051,NaN,...,-16.044551,70.543521,18.514585,0.102318,6,-2.354362,-2.354362,1.363040,False,1.134291
8,wf_2023_2024,bc_initialization,bc_initialization,0,True,"{""drawdown_control_state"": {""accuracy"": null, ...",0.909091,0.909091,0.820084,NaN,...,4.505779,7.109208,1.061804,0.013221,5,-0.154455,-2.401249,4.556762,False,0.843068
9,wf_2023_2024,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,"{""drawdown_control_state"": {""accuracy"": null, ...",0.909091,0.909091,0.820084,NaN,...,4.505779,7.109208,1.061804,0.013221,5,-0.154455,-2.401249,4.556762,False,0.843068


,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_candidate_gate,selected_loss,selected_relative_return,...,candidate_count,reference_candidate,reference_return_pct,reference_relative_return_pp,reference_max_dd_pct,reference_avg_position,reference_trade_count,selected_vs_reference_return_delta_pct,selected_vs_reference_relative_delta_pp,selected_vs_reference_max_dd_delta_pct
0,wf_2020,evaluated,bc_initialization,bc_initialization,0,False,True,False,1.501076,-15.032278,...,4,wf_pass_ytd18_15_15_02,29.156603,-30.618705,12.785207,0.171011,2,13.904689,15.586427,2.176432
1,wf_2022,evaluated,bc_initialization,bc_initialization,0,True,True,True,0.000000,71.714980,...,4,wf_pass_ytd18_15_15_02,-16.044551,70.543521,18.514585,0.102318,6,1.171459,1.171459,-2.897542
2,wf_2023_2024,evaluated,bc_initialization,bc_initialization,0,False,True,False,0.843068,4.707959,...,4,wf_pass_ytd18_15_15_02,4.505779,7.109208,1.061804,0.013221,5,-0.154455,-2.401249,4.556762


In [5]:

candidate_metrics = pd.read_csv(CARD / 'checkpoint_candidate_metrics.csv')
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')

metric_audit_cols = [
    'fold',
    'candidate',
    'metric_basis',
    'raw_metric_basis',
    'test_return',
    'raw_backtest_return',
    'metric_return_abs_diff',
    'test_buy_and_hold_return',
    'raw_backtest_buy_and_hold_return',
    'metric_buyhold_abs_diff',
    'test_relative_return',
    'raw_backtest_relative_return',
    'metric_relative_abs_diff',
    'test_max_dd',
    'raw_backtest_max_dd',
    'metric_max_dd_abs_diff',
]
metric_basis_audit = candidate_metrics[[col for col in metric_audit_cols if col in candidate_metrics.columns]].copy()
for col in ['metric_return_abs_diff', 'metric_buyhold_abs_diff', 'metric_relative_abs_diff', 'metric_max_dd_abs_diff']:
    if col in metric_basis_audit.columns:
        metric_basis_audit[col] = pd.to_numeric(metric_basis_audit[col], errors='coerce')
metric_basis_audit['material_metric_mismatch'] = metric_basis_audit.get('metric_relative_abs_diff', pd.Series(dtype=float)).fillna(0.0).ge(5.0)
metric_basis_audit.to_csv(CARD / 'metric_basis_reconciliation.csv', index=False)
material_metric_mismatches = metric_basis_audit[metric_basis_audit['material_metric_mismatch']].copy()
material_metric_mismatch_folds = sorted(material_metric_mismatches['fold'].dropna().astype(str).unique()) if not material_metric_mismatches.empty else []

evaluated = selection_by_fold[selection_by_fold['status'].eq('evaluated')].copy()
required_folds = list(WALK_FORWARD_WINDOWS)
required_fold_coverage = bool(set(required_folds).issubset(set(evaluated['fold'])) and len(evaluated) == len(required_folds))
all_candidate_gate = bool(required_fold_coverage and evaluated['selected_candidate_gate'].astype(bool).all())
any_candidate_gate = bool((not evaluated.empty) and evaluated['selected_candidate_gate'].astype(bool).any())
failed_folds = evaluated.loc[~evaluated['selected_candidate_gate'].astype(bool), 'fold'].tolist() if not evaluated.empty else required_folds
missing_folds = [fold for fold in required_folds if fold not in set(evaluated['fold'])]

if all_candidate_gate:
    decision = 'release16_cb99_checkpoint_metric_screen_passed_no_promotion'
    next_mechanism = 'train_full_retention_candidate_with_release16_cb99_warmstart_then_run_fixed_split_ytd_dashboard_and_actor_retention_gates'
elif any_candidate_gate:
    decision = 'release16_cb99_checkpoint_metric_screen_partial_pass_no_promotion'
    next_mechanism = 'inspect_failed_release16_folds_before_full_retention_training'
else:
    decision = 'release16_cb99_checkpoint_metric_screen_no_full_gate_no_promotion'
    next_mechanism = 'revise_reward_execution_or_pre2020_target_semantics_before_retention_training'

best_by_fold = {row['fold']: dict(row) for row in evaluated.to_dict('records')}
metrics_for_frontier = []
for row in evaluated.to_dict('records'):
    metrics_for_frontier.append({
        'window': row['fold'],
        'split_type': 'walk_forward_test',
        'return_pct': row.get('selected_return'),
        'max_dd_pct': row.get('selected_max_dd'),
        'relative_return_pct': row.get('selected_relative_return'),
        'avg_pos': row.get('selected_avg_pos'),
    })

drift_score_summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'required_folds': required_folds,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'required_fold_coverage': required_fold_coverage,
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'selected_candidate_gate_count': int(evaluated['selected_candidate_gate'].sum()) if not evaluated.empty else 0,
    'selected_target_gate_count': int(evaluated['selected_target_gate'].sum()) if not evaluated.empty else 0,
    'selected_drift_gate_count': int(evaluated['selected_drift_gate'].sum()) if not evaluated.empty else 0,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'selected_by_fold': best_by_fold,
    'metric_basis': 'official_daily_path_rebased_eval_window',
    'raw_metric_basis': 'backtest_profile_reported_metrics',
    'material_metric_mismatch_count': int(len(material_metric_mismatches)),
    'material_metric_mismatch_folds': material_metric_mismatch_folds,
}
write_json(CARD / 'drift_score_summary.json', drift_score_summary)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': SOURCE_CARD_NAME,
    'reference_baseline': f'{REFERENCE_EXPERIMENT}:{REFERENCE_CANDIDATE}',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'feature_frame_source': FRAME_SOURCE,
    'changed_layer': ['model_selection', 'checkpoint_selection', 'bc_warm_start'],
    'source_bc_selection': str(SOURCE_SELECTION_PATH.relative_to(ROOT)),
    'source_bc_decision': source_selection.get('decision'),
    'source_bc_selected_config': source_selection.get('selected_config'),
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': 'Diagnostic fold-local checkpoint screen only. Retention still requires a full retained actor bundle plus fixed-split, wf_2020/wf_2022/wf_2023_2024, YTD return alignment, and dashboard replay evidence.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_relative_return_by_window_pp': {fold: TARGET_RELATIVE_RETURN for fold in required_folds},
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'selected_candidate_gate_count': drift_score_summary['selected_candidate_gate_count'],
    'selected_target_gate_count': drift_score_summary['selected_target_gate_count'],
    'selected_drift_gate_count': drift_score_summary['selected_drift_gate_count'],
    'required_fold_coverage': required_fold_coverage,
    'wf_2020_coverage': bool('wf_2020' in set(evaluated['fold'])),
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'metrics': metrics_for_frontier,
    'metric_basis': 'official_daily_path_rebased_eval_window',
    'raw_metric_basis': 'backtest_profile_reported_metrics',
    'material_metric_mismatch_count': int(len(material_metric_mismatches)),
    'material_metric_mismatch_folds': material_metric_mismatch_folds,
    'metric_basis_reconciliation': 'selected metrics are recomputed from official_daily_paths; raw backtest metrics are retained only as warmup-benchmark audit columns',
    'ytd_return_alignment_evaluated': False,
    'ytd_return_alignment_blocker': 'This card trains fold-local diagnostic actors; a single retained actor must be trained before comparing YTD return to current best and SOXLBacktest references.',
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic fold-local states',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'checkpoint_candidate_metrics.csv',
        'checkpoint_selection_by_fold.csv',
        'metric_basis_reconciliation.csv',
        'official_daily_paths/',
        'drift_score_summary.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
summary = {
    **selection,
    'conclusion': [],
    'selected_by_fold': best_by_fold,
}
for row in evaluated.to_dict('records'):
    summary['conclusion'].append(
        f"{row['fold']}: selected {row['selected_candidate']} ({row['selected_kind']}) with rel={float(row['selected_relative_return']):.2f}pp, MaxDD={float(row['selected_max_dd']):.2f}%, loss={float(row['selected_loss']):.4f}, gate={bool(row['selected_candidate_gate'])}."
    )
if failed_folds:
    summary['conclusion'].append(f"Failed folds under rel >= {TARGET_RELATIVE_RETURN:.0f}pp and MaxDD < {TARGET_MAX_DD:.0f}%: {', '.join(failed_folds)}.")
else:
    summary['conclusion'].append('All required folds cleared the diagnostic checkpoint gate; promotion remains blocked until full retained-actor and YTD alignment review.')
summary['conclusion'].append(f"Metric basis: selected returns are recomputed from official daily paths; {len(material_metric_mismatches)} candidate rows have raw-vs-daily relative mismatch >= 5pp and are audited in metric_basis_reconciliation.csv.")
summary['conclusion'].append('No runtime promotion and no actor retention from this diagnostic selector.')
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "release16_cb99_rebound_checkpoint_screen_v1",
  "created_at": "2026-06-14T14:01:20.156934+00:00",
  "decision": "release16_cb99_checkpoint_metric_screen_partial_pass_no_promotion",
  "next_mechanism": "inspect_failed_release16_folds_before_full_retention_training",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "rebound_state_factor_target_repair_screen_v1",
  "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "feature_frame_source": "build_dataset_rebuilt_full_2010_2024",
  "changed_layer": [
    "model_selection",
    "checkpoint_selection",
    "bc_warm_start"
  ],
  "source_bc_selection": "runtime/model_selection/card/rebound_state_factor_target_repair_screen_v1/selection.json",
  "source_bc_decision": "rebound_state_factor_target_repair_screen_ready_for_checkpoint_no_promotion",
  "source_bc_selected_con

In [6]:

selection = json.loads((CARD / 'selection.json').read_text())
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')
metric_basis_audit = pd.read_csv(CARD / 'metric_basis_reconciliation.csv') if (CARD / 'metric_basis_reconciliation.csv').exists() else pd.DataFrame()
material_metric_mismatch_count = int(metric_basis_audit.get('material_metric_mismatch', pd.Series(dtype=bool)).astype(bool).sum()) if not metric_basis_audit.empty else 0
lines = [
    '# Release16 CB99 Rebound Checkpoint Screen',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    f"Gate: relative return >= {selection['target_relative_return_pp']:.0f}pp and MaxDD {selection['drawdown_gate_operator']} {selection['target_max_dd_pct']:.0f}% on `wf_2020`, `wf_2022`, and `wf_2023_2024`.",
    '',
    'Selected candidates:',
]
for row in selection_by_fold.itertuples(index=False):
    if row.status != 'evaluated':
        lines.append(f"- {row.fold}: {row.status}")
        continue
    lines.append(
        f"- {row.fold}: `{row.selected_candidate}` ({row.selected_kind}), rel {row.selected_relative_return:.2f}pp, "
        f"MaxDD {row.selected_max_dd:.2f}%, loss {row.selected_loss:.4f}, gate {bool(row.selected_candidate_gate)}, "
        f"vs SOXLBacktest return delta {row.selected_vs_reference_return_delta_pct:.2f}pp."
    )
lines.extend([
    '',
    f"Metric basis: selected metrics use `official_daily_path_rebased_eval_window`; `{material_metric_mismatch_count}` candidate rows have raw-vs-daily relative mismatch >= 5pp in `metric_basis_reconciliation.csv`.",
    '',
    'Risk note: selector output is diagnostic only. It retains no actor and does not override the full retained actor + fixed-split + walk-forward + YTD return-alignment promotion requirement.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `metric_basis_reconciliation.csv`, `official_daily_paths/*.csv`, `drift_score_summary.json`, `selection.json`, `summary.json`.',
])
conclusion = chr(10).join(lines).rstrip() + chr(10)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))


# Release16 CB99 Rebound Checkpoint Screen

Decision: `release16_cb99_checkpoint_metric_screen_partial_pass_no_promotion`.

Gate: relative return >= 30pp and MaxDD < 20% on `wf_2020`, `wf_2022`, and `wf_2023_2024`.

Selected candidates:
- wf_2020: `bc_initialization` (bc_initialization), rel -15.03pp, MaxDD 14.96%, loss 1.5011, gate False, vs SOXLBacktest return delta 13.90pp.
- wf_2022: `bc_initialization` (bc_initialization), rel 71.71pp, MaxDD 15.62%, loss 0.0000, gate True, vs SOXLBacktest return delta 1.17pp.
- wf_2023_2024: `bc_initialization` (bc_initialization), rel 4.71pp, MaxDD 5.62%, loss 0.8431, gate False, vs SOXLBacktest return delta -0.15pp.

Metric basis: selected metrics use `official_daily_path_rebased_eval_window`; `12` candidate rows have raw-vs-daily relative mismatch >= 5pp in `metric_basis_reconciliation.csv`.

Risk note: selector output is diagnostic only. It retains no actor and does not override the full retained actor + fixed-split + walk-forward + YTD return-alignment promotion requirement.

Next action: `inspect_failed_release16_folds_before_full_retention_training`.

Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `metric_basis_reconciliation.csv`, `official_daily_paths/*.csv`, `drift_score_summary.json`, `selection.json`, `summary.json`.
